# Neuropathy Risk Screening — ML Research Notebook

## Project Overview
This notebook documents the full machine learning pipeline for predicting peripheral neuropathy risk using a surface EMG sensor and patient demographics.

**Clinical motivation:**
Peripheral neuropathy — particularly Diabetic Peripheral Neuropathy (DPN) — causes progressive motor unit loss and changes in nerve conduction velocity. Both effects manifest directly in the EMG signal as shifts in amplitude and spectral content, making EMG a viable non-invasive screening signal.

**Task:** Binary classification — `Healthy` vs `Neuropathy Risk`

**The 4 deployment features used:**
| Feature | Source | Clinical Relevance |
|---|---|---|
| `age_years` | Patient input | Strongest independent risk factor for DPN |
| `sex` | Patient input | Hormonal differences affect neuropathy prevalence and EMG characteristics |
| `EMG_RMS_mean` | Computed from 60s recording | Captures overall amplitude of muscle electrical activity; reduced by motor unit loss |
| `EMG_median_frequency` | Computed from 60s recording | Spectral shift downward as fast-twitch fibres are preferentially lost in neuropathy |

**Dataset:** `DPN_Clinical_mechanism_and_EMG.csv` — 100,000 patient records, 19 features

**Notebook stages:**
1. Data Preprocessing
2. Exploratory Data Analysis
3. Baseline Model (DummyClassifier)
4. XGBoost with Stratified K-Fold CV
5. Model Evaluation and Export
6. Random Forest
7. Offline Prediction Function

---
## Stage 1 — Data Preprocessing

**What happens here:**
- Load the raw 100,000-row dataset
- Select only the 4 deployment-feasible features plus the target column
- Encode categorical variables (`sex`, `clinical_status`) to integers
- Stratified downsample to exactly 10,000 rows, preserving the 60/40 class ratio
- Save the cleaned dataset as `neuropathy_clean_10k.csv` — this becomes the single source of truth for all modelling

**Why only 4 features?**
The other 15 columns (blood biomarkers, clinical staging, disease duration) require laboratory tests or clinical examination. The goal is a point-of-care screening tool using only a surface EMG sensor and basic demographics.

**Why stratified downsampling?**
Training on the full 100k rows is unnecessary for this feature set and slows iteration. Stratified sampling preserves the original 60/40 class distribution so the downsampled dataset is statistically representative.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

DATASET_PATH  = '/content/DPN_Clinical_mechanism_and_EMG.csv'
CLEAN_DATA_PATH = 'neuropathy_clean_10k.csv'
TARGET_ROWS   = 10000

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATASET_PATH)
print(f"Raw dataset shape  : {df.shape}")
print(f"Columns            : {df.columns.tolist()}")

# ── Select and rename columns ─────────────────────────────────────────────────
df = df[['age_years', 'sex', 'EMG_RMS_mean', 'EMG_median_frequency', 'clinical_status']].copy()
df = df.rename(columns={'clinical_status': 'target'})
print(f"\nFiltered shape     : {df.shape}")

# ── Encode sex: Male=0, Female=1 ──────────────────────────────────────────────
df['sex'] = df['sex'].map({'Male': 0, 'Female': 1}).astype(int)
print("sex encoded        : Male=0, Female=1")

# ── Encode target: Control=0 (Healthy), all neuropathy stages=1 (Risk) ────────
df['target'] = df['target'].map({
    'Control': 0, 'mild': 1, 'moderate': 1, 'severe': 1, 'advanced': 1
}).astype(int)
print("target encoded     : Control=0, mild/moderate/severe/advanced=1")

print(f"\nClass distribution (before downsampling):")
print(df['target'].value_counts(normalize=True).to_string())

# ── Stratified downsample to 10,000 rows ─────────────────────────────────────
X_temp = df.drop('target', axis=1)
y_temp = df['target']
X_ds, _, y_ds, _ = train_test_split(
    X_temp, y_temp, train_size=TARGET_ROWS, stratify=y_temp, random_state=42
)
df_clean = pd.concat([X_ds, y_ds], axis=1).reset_index(drop=True)

print(f"\nDownsampled shape  : {df_clean.shape}")
print(f"Class distribution (after downsampling):")
print(df_clean['target'].value_counts(normalize=True).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
df_clean.to_csv(CLEAN_DATA_PATH, index=False)
print(f"\nSaved to '{CLEAN_DATA_PATH}'")
df_clean.head()

---
## Stage 2 — Exploratory Data Analysis

Before training any model, validate the dataset's integrity and understand the statistical properties of each feature. This catches encoding errors, unexpected nulls, and gives a sense of the feature distributions and relationships.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('neuropathy_clean_10k.csv')

# ── Missing values ────────────────────────────────────────────────────────────
print("=== Missing Values ===")
missing = df.isnull().sum()
if missing.any():
    print(missing[missing > 0])
else:
    print("None — dataset is complete.")

# ── Data types ────────────────────────────────────────────────────────────────
print("\n=== Data Types ===")
print(df.dtypes.to_string())

# ── Descriptive statistics ────────────────────────────────────────────────────
print("\n=== Descriptive Statistics ===")
display(df.describe().round(3))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# ── Correlation matrix ────────────────────────────────────────────────────────
plt.figure(figsize=(7, 5))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True, cmap='coolwarm', fmt='.2f',
    linewidths=0.5, square=True
)
plt.title('Feature Correlation Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.show()
print("Higher absolute values indicate stronger linear relationships with the target.")

---
## Stage 3 — Baseline Model (DummyClassifier)

A `DummyClassifier` with `strategy='most_frequent'` is trained first as a mandatory benchmarking step. It predicts the majority class — Neuropathy Risk — for every single input, ignoring all features entirely.

**Why this matters:**
Since the dataset is 60% Neuropathy Risk, always predicting that class gives 60% accuracy for free. Any real model that cannot beat this — or that only does so marginally — is not learning anything meaningful. Every subsequent model's performance is judged relative to this floor.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('neuropathy_clean_10k.csv')
X  = df.drop('target', axis=1)
y  = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("=== Baseline: DummyClassifier (Most Frequent Strategy) ===")
print(classification_report(
    y_test, y_pred_dummy,
    target_names=['Healthy', 'Neuropathy Risk'],
    zero_division=0
))
print("The 0.00 F1 for Healthy confirms the model never predicts that class.")
print("The 0.75 F1 for Neuropathy Risk is trivially achieved by always predicting it.")
print("All real models must beat both of these numbers.")

---
## Stage 4 — XGBoost with Stratified K-Fold Cross-Validation

### Model choice: XGBoost
XGBoost (Extreme Gradient Boosting) builds an ensemble of shallow decision trees sequentially. Each tree is trained on the residual errors (gradients of the log-loss) of the previous ensemble, iteratively correcting mistakes. L1 and L2 regularisation terms are embedded directly in the objective function to penalise model complexity at each boosting step.

### Why a Pipeline?
The `StandardScaler` is placed inside a scikit-learn `Pipeline` with the classifier. This is a correctness requirement, not a cosmetic choice. The scaler must be fit only on training data — if it were fit on all data before splitting, it would have seen the validation set's statistics, constituting **data leakage** that inflates performance estimates.

### Why Stratified K-Fold?
A single 80/20 split produces a performance estimate with high variance — a different random split could give a noticeably different result. K-Fold uses 5 different splits and averages across them, giving a more reliable generalisation estimate. The `Stratified` variant ensures every fold maintains the 60/40 class ratio.

**Hyperparameters:**
- `max_depth=4` — shallow trees to prevent individual trees from memorising training data
- `learning_rate=0.05` — each tree contributes only 5% of its correction; slower convergence but more robust
- `n_estimators=200` — 200 sequential boosting rounds

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

df = pd.read_csv('neuropathy_clean_10k.csv')
X  = df.drop('target', axis=1)
y  = df['target']

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb',   XGBClassifier(
        max_depth=4, learning_rate=0.05, n_estimators=200,
        random_state=42, eval_metric='logloss', verbosity=0
    ))
])

fold_accuracies  = []
y_true_all       = []
y_pred_all       = []

print("=== 5-Fold Stratified Cross-Validation — XGBoost ===")
for fold, (train_idx, test_idx) in enumerate(kf.split(X, y)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)
    print(f"  Fold {fold + 1} accuracy: {acc:.4f}")

print(f"\nMean CV Accuracy : {np.mean(fold_accuracies):.4f}")
print(f"Std across folds : {np.std(fold_accuracies):.4f}")

---
## Stage 5 — Model Evaluation and Export

### CV Test Report
The combined report below covers all 10,000 rows — each row was predicted exactly once when it was held out. This is the most honest estimate of how the model will perform on unseen data.

### Train vs Test Comparison
The training report (model predicting on data it was trained on) is compared to the CV test report. A large gap between the two indicates overfitting — the model memorised training data rather than learning generalisable patterns.

### Final Export
The pipeline is refit on the full 10,000-row dataset before saving. Cross-validation was for evaluation — for deployment, we want the model to have learned from all available data.

In [ ]:
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

df = pd.read_csv('neuropathy_clean_10k.csv')
X  = df.drop('target', axis=1)
y  = df['target']

# ── CV Test Report (combined across all 5 folds) ──────────────────────────────
print("=== XGBoost — CV Test Classification Report ===")
print(classification_report(
    y_true_all, y_pred_all, target_names=['Healthy', 'Neuropathy Risk']
))

# ── Refit on full dataset ─────────────────────────────────────────────────────
print("Fitting final pipeline on full 10,000-row dataset...")
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb',   XGBClassifier(
        max_depth=4, learning_rate=0.05, n_estimators=200,
        random_state=42, eval_metric='logloss', verbosity=0
    ))
])
final_pipeline.fit(X, y)

# ── Training Report ───────────────────────────────────────────────────────────
y_pred_train = final_pipeline.predict(X)
print("=== XGBoost — Training Classification Report ===")
print(classification_report(y, y_pred_train, target_names=['Healthy', 'Neuropathy Risk']))

# ── Overfitting check ─────────────────────────────────────────────────────────
from sklearn.metrics import classification_report as cr
train_acc = cr(y, y_pred_train, output_dict=True)['accuracy']
test_acc  = cr(y_true_all, y_pred_all, output_dict=True)['accuracy']
print(f"Train accuracy    : {train_acc:.4f}")
print(f"CV test accuracy  : {test_acc:.4f}")
print(f"Gap               : {(train_acc - test_acc)*100:.2f}% — {'acceptable' if (train_acc-test_acc) < 0.05 else 'WARNING: significant overfitting'}")

# ── Export ────────────────────────────────────────────────────────────────────
joblib.dump(final_pipeline, 'neuropathy_boosted_pipeline.pkl')
print("\nSaved to 'neuropathy_boosted_pipeline.pkl'")

---
## Stage 6 — Random Forest

A `RandomForestClassifier` is trained as a second model for comparison. Random Forest builds many independent decision trees, each trained on a different bootstrap sample of the data and using a random subset of features at each split. This dual randomisation decorrelates the trees so their errors cancel when predictions are averaged.

`class_weight='balanced'` compensates for the 60/40 class imbalance by upweighting the minority class (Healthy) during training.

This model uses a simpler 80/20 holdout split rather than K-Fold, so its test set performance estimate has higher variance than the XGBoost CV results above.

In [ ]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('neuropathy_clean_10k.csv')
X  = df.drop('target', axis=1)
y  = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest — Test Set (80/20 Split) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}\n")
print(classification_report(y_test, y_pred_rf, target_names=['Healthy', 'Neuropathy Risk']))

joblib.dump(rf, 'neuropathy_rf_model.pkl')
print("Saved to 'neuropathy_rf_model.pkl'")

---
## Stage 7 — Offline Prediction Function

`predict_neuropathy_offline()` is the end-to-end inference pipeline — it bridges the gap between a raw EMG sensor recording and a clinical verdict.

### Signal processing steps

**EMG RMS (Root Mean Square):**

$$\text{RMS} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} x_i^2}$$

Captures the overall amplitude of muscle electrical activity. Motor unit loss in neuropathy reduces the number of active fibres, altering the RMS value.

**EMG Median Frequency:**

Computed using **Welch's method** — the signal is divided into overlapping 512-sample segments, a power spectrum is estimated for each via FFT, and the spectra are averaged. The median frequency is the point where the cumulative area under the power spectrum reaches 50% of the total area. As fast-twitch motor units are preferentially lost in neuropathy, the spectrum shifts toward lower frequencies, lowering the median frequency.

### Input format
The function expects a plain text file with one voltage reading per line — exactly what an Arduino serial sketch produces.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from scipy.signal import welch
from scipy.integrate import simpson


def predict_neuropathy_offline(raw_signal_file: str,
                                age: int,
                                sex: int,
                                model_path: str = 'neuropathy_rf_model.pkl') -> str:
    """
    Predict neuropathy risk from a raw EMG signal file.

    Args:
        raw_signal_file : path to text file with one voltage reading per line
        age             : patient age in years
        sex             : 0 = Male, 1 = Female
        model_path      : path to the saved model (.pkl)

    Returns:
        'Healthy' or 'Neuropathy Risk'
    """
    # ── Load model ────────────────────────────────────────────────────────────
    if not os.path.exists(model_path):
        return f"Error: Model file not found at '{model_path}'"
    model = joblib.load(model_path)

    # ── Load raw signal ───────────────────────────────────────────────────────
    try:
        raw_data = np.loadtxt(raw_signal_file).flatten()
    except Exception as e:
        return f"Error reading signal file: {e}"

    if len(raw_data) == 0:
        return "Error: Signal file is empty"

    print(f"Samples loaded    : {len(raw_data)}")

    # ── Feature 1: EMG RMS ────────────────────────────────────────────────────
    emg_rms = float(np.sqrt(np.mean(raw_data ** 2)))

    # ── Feature 2: Median Frequency via Welch's PSD ───────────────────────────
    fs      = 1000                            # sampling frequency in Hz
    nperseg = min(len(raw_data), 512)         # window length for Welch's method
    f, Pxx  = welch(raw_data, fs=fs, nperseg=nperseg)

    total_power = simpson(Pxx, x=f)
    if total_power > 0:
        cumulative      = np.cumsum(Pxx)
        idx             = np.where(cumulative >= total_power / 2)[0]
        emg_median_freq = float(f[idx[0]]) if len(idx) > 0 else 0.0
    else:
        emg_median_freq = 0.0

    print(f"EMG RMS           : {emg_rms:.4f} µV")
    print(f"Median Frequency  : {emg_median_freq:.2f} Hz")

    # ── Predict ───────────────────────────────────────────────────────────────
    features = pd.DataFrame([{
        'age_years':            age,
        'sex':                  sex,
        'EMG_RMS_mean':         emg_rms,
        'EMG_median_frequency': emg_median_freq
    }])

    prediction  = model.predict(features)[0]
    probability = model.predict_proba(features)[0]
    verdict     = 'Neuropathy Risk' if prediction == 1 else 'Healthy'

    print(f"\nPrediction        : {verdict}")
    print(f"Healthy           : {probability[0] * 100:.1f}%")
    print(f"Neuropathy Risk   : {probability[1] * 100:.1f}%")

    return verdict


print("predict_neuropathy_offline() defined and ready.")

---
### Stage 7 (Demo) — Test Run with Synthetic Signal

A synthetic 60-second EMG signal is generated — 60,000 samples at 1000 Hz — and passed to the prediction function with sample demographic inputs. The expected result is a low-confidence prediction because the synthetic signal's statistics (very low RMS, unrealistically high median frequency) fall far outside the training data distribution. A near-50/50 probability split here is the correct behaviour — the model should be uncertain when given out-of-distribution inputs rather than giving a falsely confident wrong answer.

In [ ]:
import numpy as np

# Generate synthetic 60s signal (60,000 samples at 1000 Hz)
np.random.seed(42)
t = np.linspace(0, 60, 60000)
synthetic_signal = (
    np.random.normal(0, 0.5, 60000)
    + np.sin(2 * np.pi * 10 * t) * 0.1
    + np.random.choice([0, 1], size=60000, p=[0.99, 0.01])
    * np.random.uniform(1, 3, size=60000)
)

with open('live_test.txt', 'w') as f:
    for val in synthetic_signal:
        f.write(f"{val}\n")

print("Synthetic signal  : 60,000 samples at 1000 Hz")
print("Written to        : live_test.txt")
print("\n--- Running Prediction (Age=65, Male) ---\n")

verdict = predict_neuropathy_offline(
    raw_signal_file='live_test.txt',
    age=65,
    sex=0
)

print(f"\nFinal Verdict     : {verdict}")
print("\nNote: Near-50/50 confidence is expected here because the synthetic signal")
print("statistics (RMS ~0.54 µV, Median Freq ~488 Hz) are far outside the training")
print("distribution (RMS: 60-222 µV, Median Freq: 34-162 Hz).")